# EXP-022 겨냥 지표와 원인 분석: 회의가 다른 근거를 나눠 답하기

EXP-022 판정(보류) 뒤에 쓴다. 겨냥 지표 A(코드 집계)와 B(표본 판독)를 내고, 코드 판정에서 늘어난 인용 없는 문장과 회의 일자·위원회 불일치의 원인을 본다. 새 API 호출이 없고 저장된 기록만 읽는다. 질의 원문과 근거·답변 원문은 출력하지 않는다.

In [1]:
import json
import random
import re
from collections import Counter
from pathlib import Path

import pandas as pd

from rag.conditions import committee_base
from rag.gen_eval.checks import CITE, doc_date, sentences
from rag.gen_eval.report import per_query
from rag.index import load_cfg

In [2]:
ROOT = Path.cwd().parent
PREFIX = "hybrid-cond-filter-law-ms256-k20-ce"
JUDGE = "deepseek_go_mixed-j_v13"
RUNS = {"base": "v8-dev-full-spk-marks", "cand": "v9-dev-full-spk-marks"}
JUDGE_SUFFIX = {"base": "-s34", "cand": ""}
SEED = 20260929
N_SAMPLE = 30
SAMPLE = ROOT / "_workspace/exp022_b_sample.jsonl"
SAMPLE_KEY = ROOT / "_workspace/exp022_b_key.json"
LABELS = ROOT / "_workspace/exp022_b_labels.jsonl"
FILLERS = list(load_cfg([]).gen_eval.filler_phrases)

In [3]:
def read(path):
    with open(path, encoding="utf-8") as f:
        return {r["qid"]: r for r in map(json.loads, f)}


def cited(g):
    """답변이 인용한 근거 번호."""
    return {int(m) for m in CITE.findall(g["answer"]) if 1 <= int(m) <= len(g["contexts"])}


def year(c):
    return (doc_date(c["date"]) or str(c["date"]))[:4]  # 형식이 다른 날짜는 앞 네 글자


def names_meeting(s, nums, ctx):
    """문장 안에 인용 근거의 연도(NNNN년)나 위원회 이름이 있는가."""
    flat = s.replace(" ", "")
    return any(f"{year(ctx[i - 1])}년" in flat or committee_base(ctx[i - 1]["committee_name"]).replace(" ", "") in flat for i in nums)

## 데이터 로딩

In [4]:
gens = {k: read(ROOT / f"data/gen_eval/runs/{PREFIX}-{v}.jsonl") for k, v in RUNS.items()}
judged = {k: read(ROOT / f"data/gen_eval/judge/{PREFIX}-{v}-{JUDGE}{JUDGE_SUFFIX[k]}.jsonl") for k, v in RUNS.items()}
qids = list(gens["base"])
print({k: (len(gens[k]), len(judged[k])) for k in RUNS}, gens["base"].keys() == gens["cand"].keys())

{'base': (330, 330), 'cand': (330, 330)} True


## A. 코드 집계

여러 회의를 인용한 답변의 규모, 한 문장에 여러 연도를 함께 인용한 답변, 회의를 밝힌 문장의 비율, 인용 근거 순위를 기준과 후보에서 같은 코드로 낸다. 의미상 섞였는지는 판정하지 못한다.

In [5]:
def aggregate(g):
    n = Counter()
    for r in g.values():
        ctx = r["contexts"]
        nums = cited(r)
        n["인용한 답변"] += bool(nums)
        n["2개 이상 문서 인용"] += len({ctx[i - 1]["doc_id"] for i in nums}) >= 2
        n["2개 이상 연도 인용"] += len({year(ctx[i - 1]) for i in nums}) >= 2
        n["2개 이상 위원회 인용"] += len({committee_base(ctx[i - 1]["committee_name"]) for i in nums}) >= 2
        n["1위 근거만 인용"] += nums == {1}
        n["2위 이하도 인용"] += bool(nums - {1})
        multi = len({ctx[i - 1]["doc_id"] for i in nums}) >= 2
        mixed = False
        for s in sentences(r["answer"]):
            ns = {int(m) for m in CITE.findall(s) if 1 <= int(m) <= len(ctx)}
            if not ns:
                continue
            mixed |= len({year(ctx[i - 1]) for i in ns}) >= 2
            n["인용 문장"] += 1
            n["회의를 밝힌 인용 문장"] += names_meeting(s, ns, ctx)
            if multi:
                n["여러 문서 답변의 인용 문장"] += 1
                n["여러 문서 답변에서 회의를 밝힌 문장"] += names_meeting(s, ns, ctx)
        n["한 문장에 2개 이상 연도 인용"] += mixed
    return n


pd.DataFrame({k: aggregate(gens[k]) for k in RUNS})

,base,cand
인용한 답변,324,315
2개 이상 문서 인용,85,96
2개 이상 연도 인용,48,58
2개 이상 위원회 인용,44,48
1위 근거만 인용,237,216
2위 이하도 인용,87,99
인용 문장,726,722
회의를 밝힌 인용 문장,14,459
한 문장에 2개 이상 연도 인용,2,0
여러 문서 답변의 인용 문장,319,350


## 인용 없는 문장과 회의 일자·위원회 불일치

코드 판정에서 인용 없는 문장(33 → 68)과 불일치 답변(1 → 7)이 늘었다. 불일치가 어느 필드인지, 인용 없는 문장이 어떤 문장인지 건수로 본다.

In [6]:
def mismatch_fields(j):
    return Counter(m["field"] for r in j.values() for m in (r.get("metadata") or []))


print({k: mismatch_fields(judged[k]) for k in RUNS})
print({k: sum(bool(r.get("metadata")) for r in judged[k].values()) for k in RUNS})

{'base': Counter({'date': 1}), 'cand': Counter({'date': 4, 'committee': 4})}
{'base': 1, 'cand': 7}


In [7]:
def uncited_kinds(g, j):
    n = Counter()
    for q, r in g.items():
        if j[q]["refusal"] == "full":
            continue
        for s in sentences(r["answer"]):
            if CITE.search(s) or any(f in s for f in FILLERS):
                continue
            n["인용 없는 문장"] += 1
            n["그중 회의를 말하는 문장(회의·위원회)"] += "회의" in s or "위원회" in s
            n["그중 찾지 못함·다루지 않음"] += "찾지 못" in s or "다루고 있지 않" in s or "없습니다" in s
    return n


pd.DataFrame({k: uncited_kinds(gens[k], judged[k]) for k in RUNS})

,base,cand
인용 없는 문장,56,87
그중 회의를 말하는 문장(회의·위원회),30,65
그중 찾지 못함·다루지 않음,23,19


## 관련성 변화

관련성 차이 −0.011이 여러 문서를 인용한 답변에서 나왔는지 본다.

In [8]:
pq = {k: per_query(list(judged[k].values())) for k in RUNS}
both = pq["base"].join(pq["cand"], lsuffix="_b", rsuffix="_c")
both["multi_c"] = [len({gens["cand"][q]["contexts"][i - 1]["doc_id"] for i in cited(gens["cand"][q])}) >= 2 for q in both.index]
both["multi_b"] = [len({gens["base"][q]["contexts"][i - 1]["doc_id"] for i in cited(gens["base"][q])}) >= 2 for q in both.index]
both["d_rel"] = both["relevance_c"] - both["relevance_b"]
print(both.groupby(["multi_b", "multi_c"])["d_rel"].agg(["count", "sum", "mean"]).round(3))
print("오름", int((both.d_rel > 0).sum()), "내림", int((both.d_rel < 0).sum()), "같음", int((both.d_rel == 0).sum()))

                 count  sum   mean
multi_b multi_c                   
False   False      223 -3.5 -0.016
        True        15  0.0  0.000
True    False        4 -0.5 -0.125
        True        81  0.5  0.006
오름 19 내림 26 같음 278


## B. 판정 뒤 표본 판독: 표본 만들기

2개 이상 문서를 인용한 답변에서 기준 30개, 후보 30개를 뽑아 섞는다. 어느 실행인지 가리고, 입력은 답변과 인용한 근거이며 질의 원문은 넣지 않는다. 표본 파일은 `_workspace/`에 두고 커밋하지 않는다.

In [13]:
rng = random.Random(SEED)
rows, key = [], {}
for k in RUNS:
    pool = sorted(q for q in qids if len({gens[k][q]["contexts"][i - 1]["doc_id"] for i in cited(gens[k][q])}) >= 2)
    for q in rng.sample(pool, N_SAMPLE):
        g = gens[k][q]
        evid = [{"n": i, "date": g["contexts"][i - 1]["date"], "committee": g["contexts"][i - 1]["committee_name"], "text": g["contexts"][i - 1]["text"]} for i in sorted(cited(g))]
        rows.append({"answer": g["answer"], "evidence": evid, "_k": k, "_q": q})
rng.shuffle(rows)
if SAMPLE.exists():
    raise SystemExit(f"{SAMPLE}가 이미 있다. 판독 뒤 덮어쓰면 짝이 깨진다")
with open(SAMPLE, "w", encoding="utf-8", newline="\n") as f:
    for i, r in enumerate(rows):
        key[i] = {"run": r["_k"], "qid": r["_q"]}
        f.write(json.dumps({"id": i, "answer": r["answer"], "evidence": r["evidence"]}, ensure_ascii=False) + "\n")
SAMPLE_KEY.write_text(json.dumps(key, ensure_ascii=False), encoding="utf-8")
print(len(rows), Counter(v["run"] for v in key.values()))

60 Counter({'cand': 30, 'base': 30})


## B. 판정 뒤 표본 판독: 결과

서브에이전트 1회가 실행을 가리고 판독했다. 라벨을 실행과 대조해 묶음 비율과 Wilson 95% 구간을 낸다.

In [9]:
key = json.loads(SAMPLE_KEY.read_text(encoding="utf-8"))
with open(LABELS, encoding="utf-8") as f:
    labels = [json.loads(line) for line in f]
tab = pd.crosstab(pd.Series([key[str(r["id"])]["run"] for r in labels], name="실행"), pd.Series([r["label"] for r in labels], name="라벨"))
print(tab)


def wilson(k, n, z=1.96):
    p = k / n
    mid = (p + z * z / (2 * n)) / (1 + z * z / n)
    half = z * ((p * (1 - p) + z * z / (4 * n)) / n) ** 0.5 / (1 + z * z / n)
    return round(p, 3), round(mid - half, 3), round(mid + half, 3)


print({k: wilson(int(tab.loc[k, "묶음"]), int(tab.loc[k].sum())) for k in RUNS})

라벨    같은 사안  나눔  묶음
실행                 
base     16   8   6
cand     11  19   0
{'base': (0.2, 0.095, 0.373), 'cand': (0.0, -0.0, 0.114)}


## 결과 절의 근거 집계

인용 없는 답변의 관련성, 관련성 차이의 나뉨, 불일치 8건의 분류, 답변 형식의 재실행 변동을 건수로 낸다. 답변 원문은 출력하지 않는다.

In [10]:
nocite = [q for q in qids if not cited(gens["cand"][q]) and judged["cand"][q]["refusal"] != "full"]
print("후보 인용 없는 답변", len(nocite), Counter((judged["cand"][q].get("relevance") or {}).get("score") for q in nocite))
print("기준 인용 없는 답변", sum(not cited(gens["base"][q]) and judged["base"][q]["refusal"] != "full" for q in qids))


def n_docs(g):
    return len({g["contexts"][i - 1]["doc_id"] for i in cited(g)})


agg = Counter()
cnt = Counter()
for q in qids:
    a, b = (judged["base"][q].get("relevance") or {}).get("score"), (judged["cand"][q].get("relevance") or {}).get("score")
    if a is None or b is None:
        continue
    k = "후보 인용 없음" if n_docs(gens["cand"][q]) == 0 else "양쪽 한 문서 이하" if n_docs(gens["cand"][q]) == 1 and n_docs(gens["base"][q]) <= 1 else "그 밖"
    cnt[k] += 1
    agg[k] += b - a
print({k: (cnt[k], round(agg[k], 2)) for k in cnt})

후보 인용 없는 답변 11 Counter({1.0: 8, 0.5: 2, 0.0: 1})
기준 인용 없는 답변 0
{'그 밖': (98, 0.0), '양쪽 한 문서 이하': (215, -2.5), '후보 인용 없음': (10, -1.0)}


In [11]:
kinds = Counter()
for q, r in judged["cand"].items():
    for m in r.get("metadata") or []:
        ns = [int(x) for x in CITE.findall(m["sentence"]) if int(x) <= len(gens["cand"][q]["contexts"])]
        ctx = [gens["cand"][q]["contexts"][i - 1] for i in ns]
        if m["field"] == "committee":
            kinds["약칭(인용 근거의 이름에 포함)" if any(m["value"] in c["committee_name"] for c in ctx) else "다른 위원회"] += 1
        elif any(doc_date(c["date"]) is None for c in ctx):
            kinds["근거 날짜 형식을 읽지 못함"] += 1
        else:
            kinds["본문의 다른 날짜"] += 1
print(kinds)

Counter({'약칭(인용 근거의 이름에 포함)': 4, '본문의 다른 날짜': 2, '근거 날짜 형식을 읽지 못함': 2})


In [5]:
BULLET = re.compile(r"(?m)^[ 	]*(?:[-*•·]|[0-9]+[.)])[ 	]")
print({k: sum(bool(BULLET.search(r["answer"])) for r in gens[k].values()) for k in RUNS})
for name in ["v2-dev-full-r1", "v2-dev-full-r2", "v4-dev-full-spk-head-r1"]:
    a = read(ROOT / f"data/gen_eval/runs/{PREFIX}-{name.rsplit('-', 1)[0]}.jsonl")
    b = read(ROOT / f"data/gen_eval/runs/{PREFIX}-{name}.jsonl")
    keys = a.keys() & b.keys()
    same = sum(a[k]["answer"] == b[k]["answer"] for k in keys)
    flip = sum(bool(BULLET.search(a[k]["answer"])) != bool(BULLET.search(b[k]["answer"])) for k in keys)
    print(name, "질의", len(keys), "문구 동일", same, "불릿 유무 다름", flip)

{'base': 35, 'cand': 43}
v2-dev-full-r1 질의 100 문구 동일 20 불릿 유무 다름 3
v2-dev-full-r2 질의 100 문구 동일 19 불릿 유무 다름 2
v4-dev-full-spk-head-r1 질의 330 문구 동일 67 불릿 유무 다름 8


## 정리

- 지시는 작동했다. 회의를 밝힌 인용 문장 14/726 → 459/722, 표본 묶음 6/30 → 0/30이고 2위 이하 근거 사용은 줄지 않았다.
- 부작용은 인용이 전혀 없는 답변 0 → 11이다. 회의 일자·위원회 불일치 1 → 7답변은 코드 판정의 오탐이다(날짜 형식, 약칭, 본문의 다른 날짜).
- 사용자는 v9를 예외 채택하지 않고, 오탐을 고치고 인용 지시를 보강해 다시 재기로 정했다(D-75).